# AIC HCMC 2025 — Download + Keyframe Extraction Pipeline

Downloads `Videos_K01..K20.zip`, extracts keyframes via TransNetV2 shot detection,
saves shot intervals, and zips keyframes per `KXX` — all under `data/2025/`.

**Streaming:** each video is processed as soon as its zip is downloaded, while other
downloads continue in parallel. Two live progress bars (zip-level + video-level).

**Prerequisites:** Python 3.12, ffmpeg + aria2c in PATH (provided by the nix flake shell).

**Before running:** `Kernel → Restart & Clear Outputs`


In [ ]:
import os, subprocess, shutil, time, json, zipfile
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, wait, FIRST_COMPLETED
import numpy as np
import cv2
from tqdm.auto import tqdm

if not Path("pyproject.toml").exists() and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_CUDNN_WORKSPACE_LIMIT_IN_MB"] = "256"  # caps per-process VRAM (~1.3GB vs ~4.4GB default) so 3 GPU workers fit

# --- Paths ---
BASE_DIR = Path("data/2025")
VIDEOS_DIR = BASE_DIR / "videos"
KEYFRAMES_DIR = BASE_DIR / "keyframes"
SHOTS_DIR = BASE_DIR / "shots"
ZIPS_DIR = BASE_DIR / "keyframes_zips"
TEMP_DIR = Path("data/downloads")

for d in (VIDEOS_DIR, KEYFRAMES_DIR, SHOTS_DIR, ZIPS_DIR, TEMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Download list (K01..K20) ---
IDS = [f"K{i:02d}" for i in range(1, 21)]
VIDEO_URLS = {f"Videos_{k}.zip": f"https://aic-data.ledo.io.vn/Videos_{k}.zip" for k in IDS}

# --- Machine tuning (Ryzen 5 5600G 6C/12T / RTX 3060 12GB / 32GB) ---
ARIA_CONNECTIONS = 16
DOWNLOAD_WORKERS = 3
PROCESS_WORKERS = 6

# --- Keyframe / shot params (TransNetV2 defaults) ---
POSITIONS = [0.15, 0.5, 0.85]
SHOT_THRESHOLD = 0.5
MIN_SHOT_LEN = 15
SKIP_EXISTING = True

if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg not found. Install: nix profile install nixpkgs#ffmpeg")
if shutil.which("aria2c") is None:
    raise RuntimeError("aria2c not found. Install: nix profile install nixpkgs#aria2")

print(f"Target: {BASE_DIR.resolve()}  ({len(VIDEO_URLS)} zips)")

In [ ]:
# Verify TransNetV2 weights are real (not LFS pointers). Reinstall if corrupted.
import os
weights_dir = os.path.join(os.path.dirname(__import__("transnetv2").__file__),
                          "transnetv2-weights")
weights_pb = os.path.join(weights_dir, "saved_model.pb")
if os.path.getsize(weights_pb) < 1000:
    raise RuntimeError(
        "TransNetV2 weights are LFS pointer stubs. Run:\n"
        "  git clone --depth 1 https://github.com/soCzech/TransNetV2.git /tmp/tv2 &&\n"
        "  cd /tmp/tv2 && git lfs install --local && git lfs pull &&\n"
        f"  cp inference/transnetv2-weights/* {weights_dir} &&\n"
        f"  cp inference/transnetv2-weights/variables/* {weights_dir}/variables/")
print("TransNetV2 weights OK")

# Verify GPU + cuDNN actually work, in a SUBPROCESS so the main process stays TF-free
# (otherwise ProcessPoolExecutor's forked workers fail to set_memory_growth).
# A broken cuDNN (mixed nvidia-cudnn-cu13 from torch + nvidia-cudnn-cu12 from TF)
# otherwise fails only mid-run with "cudnn status 1002".
import subprocess, sys
check_code = (
    "import os; os.environ['TF_CPP_MIN_LOG_LEVEL']='3'\n"
    "import tensorflow as tf, numpy as np\n"
    "for g in tf.config.list_physical_devices('GPU'): tf.config.experimental.set_memory_growth(g, True)\n"
    "inp = tf.keras.Input(shape=(27, 27, 48, 3))\n"
    "tf.keras.Model(inp, tf.keras.layers.Conv3D(16, (3, 3, 3), padding='same')(inp))(np.zeros((1, 27, 27, 48, 3), dtype=np.float32))\n"
    "print('GPU_CUDNN_OK')\n"
)
r = subprocess.run([sys.executable, "-c", check_code], capture_output=True, text=True)
if "GPU_CUDNN_OK" not in r.stdout:
    raise RuntimeError(
        "GPU/cuDNN is broken (likely a mixed cuDNN install). Fix with:\n"
        "  uv pip install --reinstall --no-deps nvidia-cudnn-cu12==9.24.0.43\n"
        f"  output: {r.stdout}{r.stderr}")
print("GPU + cuDNN OK")

In [ ]:
def download_zip(name, url):
    """Download a single zip via aria2c (multi-connection). Resumes partial downloads."""
    dst = TEMP_DIR / name
    ctrl = TEMP_DIR / f"{name}.aria2"
    if dst.exists() and not ctrl.exists():
        return dst  # complete: aria2 removes the .aria2 control file on success
    subprocess.run([
        "aria2c",
        "-d", str(TEMP_DIR),
        "-o", name,
        "-x", str(ARIA_CONNECTIONS),
        "-s", str(ARIA_CONNECTIONS),
        "-c",
        "--file-allocation=none",
        "--max-tries=5",
        "--retry-wait=10",
        "--console-log-level=warn",
        "--summary-interval=30",
        url,
    ], check=True)
    return dst


def extract_zip(zip_path, target_dir):
    """Extract a ZIP into target_dir, stripping a single common top-level folder if present."""
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = [m for m in zf.namelist() if not m.endswith("/")]
        tops = {m.split("/", 1)[0] for m in names}
        strip = len(tops) == 1 and all("/" in m for m in names)
        for member in names:
            rel = member.split("/", 1)[1] if strip else member
            dest = target_dir / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(dest, "wb") as dst:
                dst.write(src.read())
    return len(names)


def get_video_fps(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    return fps if fps > 0 else 30.0


def detect_shots(model, video_path, threshold=SHOT_THRESHOLD, min_shot_len=MIN_SHOT_LEN):
    _, preds, _ = model.predict_video(str(video_path))
    preds = preds.squeeze()
    n_frames = len(preds)

    above = preds > threshold
    cuts = [0]
    i = 0
    while i < n_frames:
        if above[i]:
            j = i + 1
            while j < n_frames and above[j]:
                j += 1
            peak = i + int(np.argmax(preds[i:j]))
            if peak - cuts[-1] >= min_shot_len:
                cuts.append(peak)
            i = j
        else:
            i += 1

    shots = [(cuts[k], cuts[k + 1] - 1) for k in range(len(cuts) - 1)]
    if cuts[-1] < n_frames - 1:
        shots.append((cuts[-1], n_frames - 1))
    return [(s, e) for s, e in shots if e - s >= min_shot_len]


def compute_keyframes(shots, positions=POSITIONS):
    indices = []
    for start, end in shots:
        for pos in positions:
            idx = int(start + (end - start) * pos)
            indices.append(max(start, min(end, idx)))
    return indices


def _extract_one(args):
    video_path, timestamp, out_path = args
    tmp_path = out_path.with_suffix(".tmp.png")
    subprocess.run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-ss", str(timestamp), "-i", str(video_path),
        "-frames:v", "1", "-c:v", "png",
        "-pix_fmt", "rgb24", str(tmp_path),
    ], check=True, capture_output=True, timeout=30)
    tmp_path.replace(out_path)


def extract_frames(video_path, frame_indices, output_dir, fps):
    output_dir.mkdir(parents=True, exist_ok=True)
    tasks = []
    for fidx in frame_indices:
        out_path = output_dir / f"{fidx:06d}.png"
        if not out_path.exists() or out_path.stat().st_size == 0:
            tasks.append((str(video_path), fidx / fps, out_path))
    if tasks:
        with ThreadPoolExecutor(max_workers=min(8, len(tasks))) as pool:
            list(pool.map(_extract_one, tasks))

In [ ]:
def process_video(video_path):
    global _model
    rel = video_path.relative_to(VIDEOS_DIR)
    out_dir = KEYFRAMES_DIR / rel.with_suffix("")
    done_file = out_dir / "_done"

    if SKIP_EXISTING and done_file.exists():
        return {"video": str(rel), "skipped": True}

    out_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    shots = detect_shots(_model, str(video_path))
    t1 = time.perf_counter()

    fps = get_video_fps(str(video_path))

    shots_file = SHOTS_DIR / f"{rel.with_suffix('')}.json"
    shots_file.parent.mkdir(parents=True, exist_ok=True)
    shots_file.write_text(json.dumps({
        "fps": fps,
        "shots": [[int(s), int(e)] for s, e in shots],
        "shots_sec": [[s / fps, e / fps] for s, e in shots],
    }))

    if not shots:
        done_file.write_text(json.dumps({"shots": 0, "frames": 0}))
        return {"video": str(rel), "shots": 0, "frames": 0, "detect_s": t1 - t0}

    frame_indices = compute_keyframes(shots)
    extract_frames(str(video_path), frame_indices, out_dir, fps)
    t2 = time.perf_counter()

    done_file.write_text(json.dumps({"shots": len(shots), "frames": len(frame_indices)}))
    return {"video": str(rel), "shots": len(shots), "frames": len(frame_indices),
            "detect_s": t1 - t0, "extract_s": t2 - t1, "duration": t2 - t0}


def init_worker():
    """Load TransNetV2 once per worker process."""
    global _model
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    from transnetv2 import TransNetV2
    _model = TransNetV2()

In [ ]:
# --- Streaming: download+extract (threads) -> shot detection + keyframes (processes) ---
def download_and_extract(name, url):
    """Download + extract one zip; return extracted video paths. Idempotent."""
    loc = name.replace("Videos_", "").replace(".zip", "")
    target = VIDEOS_DIR / loc
    marker = target / ".extracted"
    if not marker.exists():
        zip_path = download_zip(name, url)
        extract_zip(zip_path, target)
        marker.touch()
    return sorted(target.rglob("*.mp4"))


t_start = time.perf_counter()
results = []
errors = []

zip_bar = tqdm(total=len(VIDEO_URLS), desc="Download+extract", unit="zip")
vid_bar = tqdm(total=0, desc="Keyframes", unit="vid")

with ProcessPoolExecutor(max_workers=PROCESS_WORKERS, initializer=init_worker) as proc_pool:
    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as dl_pool:
        pending = {}   # process future -> video path
        dl_map = {dl_pool.submit(download_and_extract, n, u): n for n, u in VIDEO_URLS.items()}
        all_futures = set(dl_map)

        while all_futures:
            done, all_futures = wait(all_futures, return_when=FIRST_COMPLETED)
            for f in done:
                if f in dl_map:
                    name = dl_map[f]
                    try:
                        videos = f.result()
                        vid_bar.total += len(videos)
                        vid_bar.refresh()
                        loc = name.replace("Videos_", "").replace(".zip", "")
                        zip_bar.write(f"  [{loc}] extracted {len(videos)} videos")
                        for v in videos:
                            pf = proc_pool.submit(process_video, v)
                            pending[pf] = v
                            all_futures.add(pf)
                    except Exception as e:
                        errors.append({"video": name, "error": str(e)})
                        zip_bar.write(f"  FAIL download {name}: {e}")
                    zip_bar.update(1)
                else:
                    v = pending.pop(f)
                    try:
                        r = f.result()
                        results.append(r)
                        if not r.get("skipped"):
                            vid_bar.set_postfix_str(f"{r['video']}: {r.get('shots', 0)} shots")
                    except Exception as e:
                        results.append({"video": str(v.relative_to(VIDEOS_DIR)), "error": str(e)})
                        vid_bar.write(f"  FAIL {v.relative_to(VIDEOS_DIR)}: {e}")
                    vid_bar.update(1)

zip_bar.close()
vid_bar.close()

t_total = time.perf_counter() - t_start

In [ ]:
# --- Summary ---
total_shots = sum(r.get("shots", 0) for r in results if "error" not in r)
total_frames = sum(r.get("frames", 0) for r in results if "error" not in r)
skipped = sum(1 for r in results if r.get("skipped"))
proc_errors = [r for r in results if "error" in r]
processed = len(results) - skipped - len(proc_errors)

print("=" * 60)
print(f"Total:  {t_total/60:.1f} min ({t_total:.0f}s)")
print(f"Videos: {len(results)} ({processed} done, {skipped} cached, {len(proc_errors)} errors)")
print(f"Shots:  {total_shots} ({total_shots/max(1, processed):.0f}/vid avg)")
print(f"Frames: {total_frames} ({total_frames/max(1, processed):.0f}/vid avg)")
print("=" * 60)
if errors or proc_errors:
    print(f"\nFailed ({len(errors) + len(proc_errors)}):")
    for r in errors:
        print(f"  [download] {r['video']}: {r['error']}")
    for r in proc_errors:
        print(f"  {r['video']}: {r['error']}")

In [ ]:
# --- Recover broken (0-byte or truncated) keyframes by re-extracting from source videos ---
import subprocess, os, cv2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

VIDEOS = Path("data/2025/videos")
KEYFRAMES = Path("data/2025/keyframes")

def find_broken():
    for dirpath, dirs, files in os.walk(KEYFRAMES):
        for fn in files:
            if fn.endswith(".png") and not fn.endswith(".tmp.png"):
                p = Path(dirpath) / fn
                try:
                    sz = p.stat().st_size
                    if sz == 0:
                        yield p
                        continue
                    with open(p, "rb") as fh:
                        fh.seek(-8, 2)
                        tail = fh.read(8)
                    if tail[0:4] != b"IEND":
                        yield p
                except OSError:
                    pass

broken = list(find_broken())
print(f"{len(broken)} broken keyframes")

if broken:
    fps_cache = {}
    for rel in {p.parent.relative_to(KEYFRAMES) for p in broken}:
        cap = cv2.VideoCapture(str(VIDEOS / f"{rel}.mp4"))
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        fps_cache[rel] = fps if fps > 0 else 30.0

    def recover(p):
        rel = p.parent.relative_to(KEYFRAMES)
        fidx = int(p.stem)
        tmp = p.with_suffix(".tmp.png")
        subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
                        "-ss", str(fidx / fps_cache[rel]), "-i", str(VIDEOS / f"{rel}.mp4"),
                        "-frames:v", "1", "-c:v", "png", str(tmp)],
                       check=True, capture_output=True, timeout=60)
        tmp.replace(p)

    with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
        list(tqdm(pool.map(recover, broken), total=len(broken), desc="Recover", unit="img"))


In [ ]:
# --- Verify all keyframes are valid PNGs ---
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("data/2025/keyframes")
SIG = b"\x89PNG\r\n\x1a\n"

broken = []
total = 0
for p in tqdm(ROOT.rglob("*.png"), desc="Verify", unit="img"):
    if p.name.endswith(".tmp.png"):
        continue
    total += 1
    try:
        with open(p, "rb") as fh:
            head = fh.read(8)
            fh.seek(-12, 2)
            tail = fh.read(12)
        if head != SIG or len(tail) < 12 or tail[4:8] != b"IEND":
            broken.append(p)
    except OSError:
        broken.append(p)

print(f"Checked {total} keyframes")
print(f"Broken: {len(broken)}")
for p in broken:
    print("  ", p)

In [ ]:
# --- Zip keyframes per KXX -> data/2025/keyframes_zips/KXX.zip (ZIP_STORED) ---
import zipfile
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

KEYFRAMES = Path("data/2025/keyframes")
OUT = Path("data/2025/keyframes_zips")

def zip_group(group):
    arc = OUT / f"{group.name}.zip"
    if arc.exists():
        return group.name, 0, arc.stat().st_size, True
    n = 0
    with zipfile.ZipFile(arc, "w", zipfile.ZIP_STORED) as z:
        for p in sorted(group.rglob("*.png")):
            if p.name.endswith(".tmp.png"):
                continue
            z.write(p, p.relative_to(KEYFRAMES))
            n += 1
    return group.name, n, arc.stat().st_size, False

OUT.mkdir(parents=True, exist_ok=True)
groups = sorted(p for p in KEYFRAMES.iterdir() if p.is_dir())
with ProcessPoolExecutor(max_workers=min(PROCESS_WORKERS, len(groups))) as ex:
    for name, n, size, was_skip in tqdm(ex.map(zip_group, groups), total=len(groups), desc="Zip keyframes", unit="zip"):
        print(f"{name}.zip: {'exists' if was_skip else f'{n} files'}, {size/1e9:.2f} GB", flush=True)